← [Overview](../00_overview.ipynb)

# Agglomerative clustering: hierarchical and contiguous

Both methods in this notebook use **Ward agglomerative clustering** — bottom-up
merging that minimises the increase in within-cluster variance at each step. They
differ only in whether merges are restricted to temporally adjacent periods.

> **Relationship to segmentation:** `contiguous` (period-level) and `SegmentConfig`
> (timestep-level within a period) are the **same algorithm** applied at different
> granularities. See [Segmentation](../06_segmentation.ipynb) for the timestep-level view.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
import plotly.io as pio
import scipy.cluster.hierarchy as sch
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

import tsam
from tsam import ClusterConfig

# tsam's *own* clustering entry point — the same function tsam.aggregate calls
# internally to turn the period matrix D into cluster labels.
from tsam.algorithms.clustering import assign_clusters

pio.renderers.default = "notebook_connected"

# Preprocessed period matrix D (normalised + unstacked) from 01_preprocessing.
tiny_period_df = pd.read_csv("../../tiny_periods.csv", header=[0, 1], index_col=0)
tiny_period_array = tiny_period_df.values  # shape (6, 8): six periods, eight features
N_PERIODS = tiny_period_array.shape[0]
N_ATTRS, N_TIMESTEPS = 2, 4
DAYS = [f"day_{p}" for p in range(N_PERIODS)]

# Raw tiny series — for the tsam.aggregate calls (it normalises internally).
tiny = pd.read_csv("../../tiny.csv", index_col=0, parse_dates=True)


print(
    "period matrix D:",
    tiny_period_array.shape,
    "  tiny:",
    tiny.shape,
)

---

## 1  Hierarchical clustering (unconstrained Ward)

**Mechanism:** bottom-up agglomeration using **Ward linkage**:

1. Start with every period in its own cluster.
2. Find the pair of clusters whose merge **increases total within-cluster
   variance the least** (Ward criterion).
3. Merge them.
4. Repeat until $k$ clusters remain.

The **Ward merge cost** between clusters $A$ and $B$ is:

$$
\Delta(A, B) = \frac{|A| \cdot |B|}{|A| + |B|} \| \bar{x}_A - \bar{x}_B \|^2
$$

where $|A|$, $|B|$ are cluster sizes and $\bar{x}_A$, $\bar{x}_B$ are their centroids.

### tsam's implementation

Ward itself is not tsam's code. `tsam.algorithms.clustering.assign_clusters` is the function
`tsam.aggregate` calls to turn the period matrix $D$ into cluster labels, and for both methods
in this notebook it hands the work to scikit-learn:

```python
if cluster_method == "hierarchical":
    clustering = AgglomerativeClustering(n_clusters=n_clusters, linkage="ward")
else:  # contiguous: only adjacent periods may be merged
    adjacency_matrix = np.eye(len(candidates), k=1) + np.eye(len(candidates), k=-1)
    clustering = AgglomerativeClustering(
        n_clusters=n_clusters, linkage="ward", connectivity=adjacency_matrix
    )
return np.asarray(clustering.fit_predict(candidates))
```

So this notebook does **not** re-derive Ward. It takes the algorithm as given and shows the
three things that matter when you use it: **what goes in** (§1.1), **what happens in between**
(§1.2), and **what comes back** (§1.3).

**TSAM configuration for hierarchical clustering:**

In [ ]:
# Hierarchical: bottom-up Ward agglomerative clustering, unconstrained.
# representation defaults to 'medoid' for hierarchical.
cfg_hierarchical = ClusterConfig(method="hierarchical", representation="medoid")
print(cfg_hierarchical)

# Full aggregate call:
# result = tsam.aggregate(
#     df,
#     n_clusters=k,
#     period_duration="1D",
#     cluster=cfg_hierarchical,
# )

### 1.1  In — what Ward receives

Exactly one thing: the the input period matrtix $D$, a plain `(6, 8)` array of floats. One row per
period (6), one column per (attribute (2), timestep (4))(8) coordinate.


In [ ]:
# The input, as a human reads it (labelled) ...
print(
    "D — the one and only input to Ward:",
    tiny_period_array.shape,
    "= (periods, attributes x timesteps)\n",
)
display(tiny_period_df.round(4))

# ... and as scikit-learn receives it: an anonymous float array, no time, no names.
print("\nWhat scikit-learn actually gets (tiny_period_array):")
print(tiny_period_array.round(4))

### 1.2  Middle — the merge history

`assign_clusters` asks for a fixed `n_clusters`, so it only ever sees **one cut** of the tree
and throws the rest away. But the estimator computes the whole thing, and if you fit it with
`distance_threshold=0` instead, it keeps every merge and exposes the history on two attributes:

* **`children_`** — the pair merged at each step. Ids `0…5` are the original days; id `6 + j`
  is the cluster *created* at step `j`. So the tree is written in terms of its own history.
* **`distances_`** — the Ward cost $\Delta(A, B)$ of that merge, i.e. how much within-cluster
  variance it cost to join the pair.

Five merges take six singleton clusters down to one. That table is the entire intermediate
state of the algorithm — everything else in this section is a different rendering of it.

In [ ]:
# The same estimator assign_clusters builds, but fitted to keep *every* merge
# (distance_threshold=0) instead of stopping at k. Same data, same Ward criterion —
# only the stopping point differs.
tree = AgglomerativeClustering(
    n_clusters=None, distance_threshold=0, linkage="ward", compute_distances=True
).fit(tiny_period_array)


# Unroll that history into a readable table. `inside[i]` = the days sitting in cluster id i.
days_inside_column = {p: {p} for p in range(N_PERIODS)}
history = []
for step, ((period_1_merged, period_2_merged), cost) in enumerate(
    zip(tree.children_, tree.distances_)
):
    new_id = N_PERIODS + step  # ids 0..5 are days; 6+ are clusters born here
    days_inside_column[new_id] = (
        days_inside_column[period_1_merged] | days_inside_column[period_2_merged]
    )
    history.append(
        {
            "step": step,
            "merged": f"{period_1_merged} + {period_2_merged}",
            "ward cost": round(float(cost), 4),
            "-> new cluster id": new_id,
            "days inside it": sorted(days_inside_column[new_id]),
            "clusters left": N_PERIODS - step - 1,
        }
    )

print()
pd.DataFrame(history).set_index("step")

**Reading the history.** Ward merges the two sunny days first (cost 0.23), then the two
overcast days (0.30), then the two cloudy days (0.49) — the three shape-pairs, cheapest first.
Only then, having run out of cheap merges, does it start joining *pairs* to each other, and the
price jumps: 0.49 → 1.03. That jump is the argument for cutting at **k=3**. Below it, merges are
nearly free because the days really are alike; above it, every further merge forces genuinely
different days together.

The view that follows is the same five rows.

In [ ]:
# View 1 — the dendrogram: merge cost on the y-axis, so the k=3 jump is the tall gap.
#
# scipy renders the picture; the tree it draws is the same one sklearn just built.
linkage = sch.ward(tiny_period_array)
print(
    "scipy's tree == the sklearn tree tsam uses:",
    np.allclose(linkage[:, :3], np.column_stack([tree.children_, tree.distances_])),
)

fig_dendro = ff.create_dendrogram(
    tiny_period_array,
    labels=DAYS,
    linkagefun=lambda _pdist: linkage,
    colorscale=px.colors.qualitative.Set1[:6],
)
fig_dendro.update_layout(
    title=(
        "Ward dendrogram — the merge history as a tree<br>"
        "<sup>Bar height = ward cost. The three shape-pairs join cheaply near the "
        "floor; the tall gap above them is where k=3 sits.</sup>"
    ),
    yaxis_title="Ward cost of the merge",
    xaxis_title="Period",
)
fig_dendro.show()

### 1.3  Out — what comes back

One array of six integers: `labels[p]` is the cluster of day `p`. That is the whole output
contract of the clustering step — the tree, the costs and the merge order are all discarded,
and only the chosen row of the heatmap above survives into the rest of the pipeline.

Below, the `k=3` row is obtained three ways: by cutting the tree we just drew, by calling
tsam's `assign_clusters` on $D$, and by running the full `tsam.aggregate` pipeline on the raw
series. The three must agree, and they do.

In [ ]:
# (a) The tree above, cut at k=3.
labels_from_tree = sch.fcluster(linkage, t=3, criterion="maxclust")

# (b) tsam's clustering step, called directly on the period matrix D.
labels_assign = assign_clusters(
    tiny_period_array, n_clusters=3, cluster_method="hierarchical"
)

# (c) The full pipeline: tsam.aggregate on the raw series (it builds D itself).
result_hier_tiny = tsam.aggregate(
    tiny,
    n_clusters=3,
    period_duration="1D",
    cluster=ClusterConfig(method="hierarchical"),
)
labels_aggregate = np.asarray(result_hier_tiny.cluster_assignments)

print("(a) tree cut at k=3      :", labels_from_tree)
print("(b) assign_clusters(D, 3):", labels_assign)
print("(c) tsam.aggregate       :", labels_aggregate)
print("\nCluster counts:", result_hier_tiny.cluster_counts)

# The cluster *numbers* differ; the *grouping* must not. A rand score of 1.0 means the two
# labellings partition the days identically, whatever the groups happen to be called.
print(
    "\n(a) vs (b) identical partition:",
    adjusted_rand_score(labels_from_tree, labels_assign) == 1.0,
)
print(
    "(b) vs (c) identical partition:",
    adjusted_rand_score(labels_assign, labels_aggregate) == 1.0,
)

**Reading the assignment array.** Entry `p` is the cluster of tiny **day `p`**. All three
routes group days 0–1 (sunny), 2–3 (overcast) and 4–5 (cloudy) — the same three shape-pairs
that k-means and k-medoids find in the
[partitional notebook](01_partitional_clustering.ipynb), reached here by a completely
different route: cheapest-merge-first, as the history table showed.

The cluster **numbers** differ between the routes (`fcluster` numbers by tree position,
scikit-learn by merge order), which is why the check uses the rand score rather than `==`:
what a clustering asserts is **which days share a cluster**, never what the group is called.
Downstream code must never depend on the numbering.

---

## 2  Contiguous clustering — Ward with a temporal-adjacency constraint

> **Common misconception corrected:** `contiguous` is sometimes described as
> "time-based". This is wrong. Algorithmically it is **Ward agglomerative
> clustering with an adjacency-matrix connectivity constraint** — identical to
> `hierarchical`, except the connectivity matrix restricts merges to
> **immediately adjacent periods** only. It is therefore:
>
> * **Feature-based** (uses Ward / within-cluster variance, i.e. feature similarity)
> * **With a time-contiguity constraint** (can only merge neighbours)
>
> It sits in the **feature-based** row of the Hoffmann taxonomy.

Section 1.1 made the point that the input period matrtix $D$ carries no time information directly, so Ward cannot
possibly favour neighbouring days. Time therefore has to arrive as a **second input**: the
bidiagonal adjacency matrix $I(|i-j| = 1)$, passed to the *same* estimator as `connectivity`.
That single extra argument is the entire difference between the two methods — same $D$, same
Ward criterion, same shape of output. Only the set of merges Ward is *allowed* to consider
changes.

**Connection to segmentation:** `contiguous` and
[segmentation](../06_segmentation.ipynb) are the **same algorithm**
(Ward + adjacency constraint) applied at different granularities:
* `contiguous` merges **periods** (rows of the D matrix)
* segmentation merges **timesteps** within each period

**TSAM configuration for contiguous clustering:**

In [ ]:
# Contiguous: Ward agglomerative clustering with temporal-adjacency constraint.
# Feature-based (Ward variance criterion), not time-based.
# representation defaults to 'medoid' for contiguous.
cfg_contiguous = ClusterConfig(method="contiguous", representation="medoid")
print(cfg_contiguous)

# Full aggregate call:
# result = tsam.aggregate(
#     df,
#     n_clusters=k,
#     period_duration="1D",
#     cluster=cfg_contiguous,
# )

In [ ]:
# The second input, exactly as assign_clusters builds it (clustering.py):
adj = np.eye(N_PERIODS, k=1) + np.eye(N_PERIODS, k=-1)  # bidiagonal: I(|i - j| = 1)

print("In (1): D, unchanged  ", tiny_period_array.shape)
print(
    "In (2): adjacency     ", adj.shape, "— 1 where two periods are neighbours in time"
)
print(pd.DataFrame(adj.astype(int), index=DAYS, columns=DAYS), "\n")

# Same function, same D, one word changed: cluster_method="contiguous". Internally that
# passes `adj` to AgglomerativeClustering as `connectivity` and changes nothing else.
labels_contiguous = assign_clusters(
    tiny_period_array, n_clusters=3, cluster_method="contiguous"
)
print("Out: same shape as before —", labels_contiguous)

In [ ]:
# Tiny set: contiguous through the full pipeline.
result_cont_tiny = tsam.aggregate(
    tiny,
    n_clusters=3,
    period_duration="1D",
    cluster=ClusterConfig(method="contiguous"),
)
print("Contiguous   (tiny):", np.asarray(result_cont_tiny.cluster_assignments))
print("Hierarchical (tiny):", np.asarray(result_hier_tiny.cluster_assignments))
print("-> constraint does not bind: same partition as unconstrained Ward.\n")

**Why the tiny set cannot show the difference.** Its three shape-groups — sunny (0,1),
overcast (2,3), cloudy (4,5) — happen to fall on *consecutive* days, so the merges
unconstrained Ward wants to make are already merges of neighbours. The adjacency constraint
forbids nothing, and both methods return the same partition. A dataset only distinguishes the
two methods when similar periods are **scattered** across the calendar.


---

**Up next:**
* [Extremal-prototype selection](03_extremal_prototype_selection.ipynb) — k-maxoids, the spread-maximising method

**See also:**
* [Averaging](04_averaging.ipynb) — the one time-based grouping method (positional blocks)
* [Segmentation](../06_segmentation.ipynb) — the timestep-level analogue of `contiguous`